# Azure OpenAI Chat Completion Demo

This notebook demonstrates how to call an Azure OpenAI chat completion model (`gpt-5.4-mini`) from Python.

It covers:

1. Installing the SDK
2. Setting up the Azure OpenAI client
3. Sending a single chat completion request
4. Maintaining chat history across multiple turns
5. Basic input validation and error handling

Requirements: Python 3.12.10 or later, VS Code with the Jupyter extension.

## Step 1: Install the OpenAI SDK

Run this once per environment. If you already have it installed, you can skip this cell.

In [ ]:
# Installs the OpenAI Python SDK (includes the Azure client)
%pip install --upgrade openai

## Step 2: Import dependencies

In [ ]:
# Azure-specific OpenAI client
from openai import AzureOpenAI

## Step 3: Configure your Azure OpenAI connection

Paste your own endpoint and key below. `gpt-5.4-mini` is set as the deployment name directly since that is the model deployed for this lab.

Note: this hardcodes the key directly in the notebook, which is fine for a local training exercise but should not be done in shared or production notebooks. Never commit a notebook with a real key in it to a public repository.

In [ ]:
# Paste your Azure OpenAI endpoint and key here
azure_oai_endpoint = "https://hakunamatata1.services.ai.azure.com/"
azure_oai_key = "<REPLACE_WITH_YOUR_API_KEY>"

# Deployment name is fixed for this lab
azure_oai_deployment = "gpt-5.4-mini"

# Fails fast with a clear message instead of a confusing error later on
if not azure_oai_endpoint or "<" in azure_oai_endpoint or not azure_oai_endpoint.startswith("https://"):
    raise ValueError("Replace the placeholder endpoint above with your real Azure OpenAI endpoint before continuing.")

if not azure_oai_key or "<" in azure_oai_key or len(azure_oai_key) < 10:
    raise ValueError("Replace the placeholder key above with your real Azure OpenAI key before continuing.")

print("Configuration loaded.")

## Step 4: Initialize the Azure OpenAI client

In [ ]:
# Creates the client used for every request in this notebook
client = AzureOpenAI(
    azure_endpoint=azure_oai_endpoint,
    api_key=azure_oai_key,
    api_version="2024-12-01-preview",  # supports newer model deployments; adjust if your resource pins an older GA version
)

## Step 5: Send a single chat completion request

This is the core demo: a single call to the model with a short conversation, showing how system, user, and assistant roles work together.

In [ ]:
# Sends a one-off chat completion request with a short example conversation
response = client.chat.completions.create(
    model=azure_oai_deployment,
    temperature=0,
    max_completion_tokens=100,
    messages=[
        {"role": "system", "content": "You are a helpful assistant with a lot of humor."},
        {"role": "user", "content": "Do penguins like pizzas?"},
        {"role": "assistant", "content": "Yes, penguins like pizza very much. They also like chips and hummus."},
        {"role": "user", "content": "Do polar bears support pizzas too?"},
    ],
)

In [ ]:
# Extracts the reply text and guards against an empty response
generated_text = response.choices[0].message.content if response.choices else None

if not generated_text:
    print("No response was returned by the model.")
else:
    print("Response: " + generated_text + "\n")

## Step 6: Maintain chat history

To hold a real conversation, keep a running list of messages and append every user and assistant turn to it before the next request.

In [ ]:
# Starts the conversation history with a system prompt and one prior exchange
messages_array = [
    {"role": "system", "content": "You are a helpful assistant with a lot of humor."},
    {"role": "user", "content": "Do penguins like pizzas?"},
    {"role": "assistant", "content": "Yes, penguins like pizza very much. They also like chips and hummus."},
]

In [ ]:
# Small helper that sends the current history to the model and returns the reply
def ask_model(messages, temperature=0.7, max_completion_tokens=100):
    response = client.chat.completions.create(
        model=azure_oai_deployment,
        temperature=temperature,
        max_completion_tokens=max_completion_tokens,
        messages=messages,
    )

    if not response.choices:
        return None

    return response.choices[0].message.content

## Step 7: Install Gradio

Install Gradio, a lightweight Python library for quickly creating web-based user interfaces.


In [ ]:
%pip install --upgrade gradio

## Step 8: Polished web UI with Gradio (with live history)

Gradio launches a small local web chat UI in a couple of lines, embedded right in the notebook output. It runs the same `ask_model()` function underneath, and it doubles as Step 8: instead of printing `messages_array` manually after the fact, the history is rendered live, right below the chat, and updates itself after every message.

In [ ]:
import gradio as gr


def gradio_chat(user_message, history):

    if not user_message or not user_message.strip():
        return "Please type a message before sending."

    if user_message.lower().strip() in ["quit", "exit"]:
        return (
            "Session ended. "
            "You may close this browser tab or refresh the page."
        )

    conversation = [
        {
            "role": "system",
            "content": (
                "You are a helpful assistant with a lot of humor."
            ),
        }
    ]

    for turn in history:

        if isinstance(turn, dict):
            conversation.append(turn)

        else:
            past_user_msg, past_assistant_msg = turn

            conversation.append(
                {
                    "role": "user",
                    "content": past_user_msg,
                }
            )

            if past_assistant_msg:
                conversation.append(
                    {
                        "role": "assistant",
                        "content": past_assistant_msg,
                    }
                )

    conversation.append(
        {
            "role": "user",
            "content": user_message,
        }
    )

    generated_text = ask_model(
        conversation,
        temperature=0.7,
        max_completion_tokens=100,
    )

    if not generated_text:
        return "No response was returned by the model."

    return generated_text


with gr.Blocks(
    title="Azure AI Chat",
    theme=gr.themes.Soft()
) as demo:

    with gr.Row():

        gr.Markdown(
            """
<div style="max-width: 900px; margin: 0 auto;">

# Azure AI Chat

Welcome to Azure AI Chat.

### Notes

- Chat history is maintained during this session.
- Type **exit** or **quit** to end the session.
- Refresh the page to start a new conversation.

</div>
"""
        )

    with gr.Row():

        with gr.Column(
            scale=1,
            min_width=320
        ):
            gr.ChatInterface(
                fn=gradio_chat,
                textbox=gr.Textbox(
                    placeholder="Type a message... (type 'exit' to end the session)",
                    container=True
                )
            )


print("\nOpen in browser:")
print("http://127.0.0.1:7860\n")

demo.launch(
    inline=False,
    inbrowser=False,
    share=False
)

## Conclusion

Nice work, here's what you just learned:

* How to set up the Azure OpenAI client with an endpoint, key, and deployment name.
* How to send a single chat completion request and read the reply from `response.choices[0].message.content`.
* How to turn that into a real, multi-turn conversation using a `messages_array` that you keep appending to.
* How to wrap that logic in a Gradio chat UI, so instead of typing into a plain input box, you get a proper chat window with live-updating history underneath it.

### Where this shows up in real life

This is not just a notebook exercise. The exact `messages_array` pattern powering your Gradio chat is the backbone of every agentic AI system out there. Once you start building agents, this is the same conversation history you will pass back into the model on every turn, along with tool calls, tool results, and intermediate reasoning steps. If you understand how to grow that array correctly here, you already understand the core loop that every agent framework is doing under the hood.

### A tip before you go

Never hardcode your endpoint and key directly in a notebook like we did in Step 3. That was fine for this lab, but in any real project:

* Put your secrets in a `.env` file (`AZURE_OPENAI_ENDPOINT`, `AZURE_OPENAI_API_KEY`, `AZURE_OPENAI_DEPLOYMENT`).
* Load them with `python-dotenv` (`pip install python-dotenv`, then `load_dotenv()` and `os.getenv("AZURE_OPENAI_API_KEY")`).
* Add `.env` to your `.gitignore` so it never gets pushed to a repository.

That one habit alone will save you from ever leaking a key by accident.

### Try next

* Add a `quit`/`exit` message inside `gradio_chat` that closes the Gradio server (`demo.close()`) instead of leaving it running.
* Save `messages_array` to a file so a conversation can be resumed later.
* Swap the system prompt to change the assistant's behaviour for a different use case.